# 02 — Réseau de neurones (MLP)

## Rôle dans le projet
On prédit le **débit de chaque capteur 15 min plus tard (t+1)** à partir de l'information disponible à l'instant t. C'est une **régression supervisée**. Ce notebook teste le modèle « réseau de neurones » ; les autres familles de modèles sont comparées dans les autres notebooks avec le même protocole et les mêmes métriques.

Données : `traffic_dataset_t1.npz`, généré par `src/make_dataset.py`. X est identique au `.mat` d'origine (48 variables par capteur et par instant : 10 débits passés t−9…t, jour, heure, nombre de voies, direction, route). Seule la cible change : dans le `.mat`, `tra_Y` vaut en réalité le débit à **t+2** (vérifié sur 100 % des lignes) alors que l'énoncé du dataset annonce t+1, donc on la recalcule. Chaque exemple est un couple (instant, capteur) : 1261 × 36 ≈ 45 000 exemples d'entraînement.

## Lien avec le cours (chapitre 3)
- **Architecture** : perceptron multicouche à **une couche cachée** : 48 entrées normalisées → 30 neurones `tanh` → 1 neurone de sortie **linéaire** (régression : la sortie ne doit pas être bornée).
- **Pourquoi une activation non linéaire** : sans elle, le réseau se réduit à une régression linéaire, quel que soit le nombre de couches.
- **Pourquoi une seule couche cachée** : le théorème d'approximation universelle dit qu'elle suffit en principe. Il ne dit rien du nombre de neurones, d'où l'étude de ce réglage plus bas.
- **Apprentissage** : minimisation de l'erreur quadratique par rétropropagation (optimiseur Adam de scikit-learn). Le problème n'est pas convexe, donc le résultat dépend de l'initialisation : c'est pour ça qu'on répète chaque configuration avec plusieurs graines.
- **Surapprentissage** : contrôlé par **early stopping** sur un jeu de validation.
- **Taille du modèle** : Σ (entrées + 1) × neurones = (48+1)×30 + (30+1)×1 = **1501 paramètres** (1801 avec les 58 entrées de la variante spatiale), à comparer aux ~45 000 exemples.

## Protocole
- Découpage **chronologique** (`src/data.py`) : train = 80 % premiers instants du jeu d'entraînement, val = 20 % suivants, test = jeu de test d'origine, utilisé **une seule fois** à la fin.
- Normalisation (`StandardScaler`) apprise sur le train uniquement.
- Métriques : RMSE et MAE, en unités de débit normalisé (0 à 1).
- Référence à battre : la **persistance** (débit à t+1 = débit à t).

## Pistes testées
- **spatial** : ajoute la moyenne des 10 débits passés des capteurs voisins (matrice d'adjacence `A`) → 58 entrées. Idée : un bouchon en amont arrive au capteur quelques minutes plus tard.
- **variation** : le modèle apprend `y - x_t` (ce que la persistance ne sait pas prédire), puis on rajoute `x_t`. La cible est alors centrée sur 0 et plus facile à apprendre.
- Chaque config est entraînée avec plusieurs graines : moyenne ± écart-type des scores, et score de l'**ensemble** (moyenne des prédictions des différents modèles). Même idée que le bagging des forêts aléatoires : moyenner des modèles différents réduit la variance.
- Early stopping patient (`tol=1e-6`, `n_iter_no_change=30`) : le `tol=1e-4` par défaut arrête trop tôt.

## Préparation
Imports, et génération du dataset t+1 s'il n'existe pas encore (il n'est pas versionné dans git).

In [ ]:
import sys; sys.path.append("..")
from functools import cache
import numpy as np
import matplotlib.pyplot as plt
from src.data import build, DATA_PATH, T1_PATH
from src.make_dataset import main as make_dataset
from src.utils import evaluate
from sklearn.neural_network import MLPRegressor

if not T1_PATH.exists():  # dataset non versionné (.gitignore) : on le génère
    make_dataset()

SEEDS = range(5)

### Fonctions utilitaires
- `data(variant, path)` : charge le jeu normalisé et non normalisé (le second sert à récupérer `x_t` en unités d'origine pour la variante « variation »).
- `mlp(...)` : le modèle décrit plus haut ; `h` = nombre de neurones cachés, `act` = activation.
- `run(...)` : entraîne un MLP par graine et renvoie la RMSE moyenne, son écart-type, et les scores de l'ensemble. Avec `split="val"`, on entraîne sur train et on évalue sur val (choix des réglages) ; avec `split="test"`, on réentraîne sur train+val et on évalue sur le test (score final). `path` choisit le fichier source : `T1_PATH` (cible t+1, par défaut) ou `DATA_PATH` (`.mat` d'origine, cible t+2).

In [ ]:
@cache
def data(variant, path=T1_PATH):
    d = build(variant=variant, scaled=True, path=path)
    raw = build(variant=variant, scaled=False, path=path)  # pour récupérer x_t non normalisé
    return d, raw


def mlp(h=30, act="tanh", seed=0):
    return MLPRegressor(hidden_layer_sizes=(h,), activation=act, early_stopping=True,
                        tol=1e-6, n_iter_no_change=30, max_iter=500, random_state=seed)


def run(variant="base", cible="debit", split="val", h=30, act="tanh", path=T1_PATH):
    """Entraîne un MLP par graine, renvoie moyenne et écart-type des RMSE, et le score de l'ensemble."""
    d, raw = data(variant, path)
    tr, ev = ("train", "val") if split == "val" else ("trainval", "test")
    X, y, X_ev, y_ev = d[f"X_{tr}"], d[f"y_{tr}"], d[f"X_{ev}"], d[f"y_{ev}"]
    x_t, x_t_ev = raw[f"X_{tr}"][:, 9], raw[f"X_{ev}"][:, 9]
    if cible == "variation":
        y = y - x_t
    scores, preds = [], []
    for s in SEEDS:
        p = mlp(h, act, s).fit(X, y).predict(X_ev)
        if cible == "variation":
            p = p + x_t_ev
        scores.append(evaluate(y_ev, p)["RMSE"])
        preds.append(p)
    return np.mean(scores), np.std(scores), evaluate(y_ev, np.mean(preds, axis=0))


def show(name, res):
    moy, std, ens = res
    print(f"{name:28} RMSE {moy:.4f} ± {std:.4f} | ensemble MAE {ens['MAE']:.4f} RMSE {ens['RMSE']:.4f}")

## Référence naïve : débit à t+1 = débit à t
À 15 min d'horizon le trafic varie peu, donc cette référence est déjà forte. Un modèle n'a d'intérêt que s'il fait **mieux que ce score**.

In [ ]:
d, raw = data("base")
evaluate(raw["y_val"], raw["X_val"][:, 9])

## Courbe d'apprentissage (modèle simple, une graine)
Perte sur le train et R² sur les 10 % mis de côté par l'early stopping, à chaque epoch (un passage complet sur les données).

À lire : la perte d'entraînement baisse toujours ; l'apprentissage s'arrête quand le score de validation cesse de s'améliorer. C'est l'early stopping du cours.

*Remarque* : scikit-learn tire ces 10 % **au hasard** parmi les lignes, donc des instants voisins tombent à la fois en apprentissage et en validation interne. Ce score est un peu optimiste ; les comparaisons de modèles se font donc sur notre jeu de validation chronologique, pas sur cette courbe.

In [ ]:
model = mlp().fit(d["X_train"], d["y_train"])
print(f"{model.n_iter_} epochs, meilleure epoch : {np.argmax(model.validation_scores_) + 1}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.plot(model.loss_curve_)
ax1.set(xlabel="epoch", ylabel="perte (train)", yscale="log")
ax2.plot(model.validation_scores_)
ax2.set(xlabel="epoch", ylabel="R² (validation interne)", ylim=(0.95, 1))
plt.tight_layout()

## Base / spatial × débit / variation (30 neurones, tanh)
On croise les deux pistes. Ces scores sont calculés sur le jeu de **validation**, pour choisir la configuration sans regarder le test. À comparer à la persistance ci-dessus.

In [ ]:
for variant in ["base", "spatial"]:
    for cible in ["debit", "variation"]:
        show(f"{variant}, {cible}", run(variant, cible))

## Nombre de neurones et activation (spatial, variation)
Réglage de la capacité du réseau. Trop peu de neurones : le modèle est trop simple (biais). Trop : il peut surapprendre (variance), même si l'early stopping limite ce risque. On compare aussi `tanh` (activation du TD1) et `relu` (plus courante aujourd'hui, pas de saturation du gradient).

In [ ]:
for act in ["tanh", "relu"]:
    for h in [10, 30, 50, 100]:
        show(f"h={h}, {act}", run("spatial", "variation", h=h, act=act))

## Évaluation finale sur le test
Config retenue, réentraînée sur train+val, puis évaluée **une seule fois** sur le test : c'est le score à reporter dans le rapport, à côté de la persistance sur le même jeu.

In [ ]:
d, raw = data("base")
print("persistance", evaluate(raw["y_test"], raw["X_test"][:, 9]))
show("spatial, variation (test)", run("spatial", "variation", split="test"))

## Horizon t+2 (30 min)
Même modèle, mais pour prédire le débit **30 min plus tard**. Pas besoin de nouveau dataset : la cible du `.mat` d'origine est justement à t+2 (`DATA_PATH`). X est identique, seul le label change.

On garde la configuration retenue à t+1 (spatial, variation, 30 neurones tanh) sans la re-régler, et la référence devient la persistance à 30 min : ŷ(t+2) = y(t). On s'attend à une erreur plus grande pour tous les modèles (l'avenir est moins prévisible), mais à un écart plus net avec la persistance, puisque le trafic a plus le temps de varier.

In [ ]:
for split in ["val", "test"]:
    d2, raw2 = data("base", DATA_PATH)
    print(f"--- {split}")
    print("persistance t+2", evaluate(raw2[f"y_{split}"], raw2[f"X_{split}"][:, 9]))
    show(f"spatial, variation t+2", run("spatial", "variation", split=split, path=DATA_PATH))

## Limites et perspectives
- Le MLP traite chaque (instant, capteur) indépendamment : il ne voit l'historique qu'à travers les 10 débits passés fournis en entrée, et la topologie routière seulement via la moyenne des voisins.
- Pistes non traitées faute de temps : réseaux récurrents (LSTM) sur la série brute, réseaux de neurones sur graphe exploitant directement `A`, prédiction au-delà de 30 min.